# Kapitel 3 — Ein Modell anpassen · Notebook zum Ausfüllen

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch03_a_completer.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Die zu bearbeitenden Zellen sind mit **ZU ERGÄNZEN** markiert. Führen Sie die Zellen der Reihe nach aus: Solange eine Antwort fehlt, melden die Überprüfungen dies ohne Fehler. Die Lösung finden Sie in `ch03_corrige.ipynb` auf derselben Begleitseite.

**Verwendeter Verlust**: der mittlere quadratische Fehler (Gleichung 3.2) auf den Wohnungen A bis F, sofern nicht anders angegeben.

**Inhalt.** Daten · Die Rechnungen der Lektionen 3.1 bis 3.6 · Aufgaben 3.1 bis 3.6

*Die Zelle von Lektion 3.4 und die von Aufgabe 3.4 f) führen Hunderttausende von Schritten aus: Rechnen Sie jeweils mit einigen Sekunden.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Farben des Buchs: eine Farbe = eine Rolle (beobachtet, vorhergesagt, Abweichung, Gerade des Kollegen)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Daten

Die Kennung D0.1 ist die des Buchs und der CSV-Datei.

In [ ]:
# Daten von Kapitel 3, aus der CSV-Datei des Buchs übernommen (nicht ändern)

# D0.1 — die sechs Wohnungen aus der Einleitung, für die Anpassung verwendet
homes = pd.DataFrame({
    'home':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'floor_area': [32, 45, 54, 70, 85, 98],
    'price':  [118, 172, 185, 245, 279, 335],
})
x = homes['floor_area'].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)

# D3.1 — vier neue Verkäufe, nur zur Bewertung verwendet (Lektion 3.6)
new_sales = pd.DataFrame({
    'sale':      ['V1', 'V2', 'V3', 'V4'],
    'floor_area': [38, 58, 76, 92],
    'price':  [132, 215, 254, 322],
})
xv = new_sales['floor_area'].to_numpy(dtype=float)
yv = new_sales['price'].to_numpy(dtype=float)

# Wohnung G (Einleitung): Rechenbeispiel, nie für die Anpassung verwendet
x_G, y_G = 62, 209

### Die Variablen der Daten

Die Namen im Code sind englisch. Für die Spalten gibt die letzte Spalte den Namen an, der in den CSV-Dateien des Buchs verwendet wird.

| Name | Bedeutung | Einheit | Spalte der CSV-Datei |
|:--|:--|:--|:--|
| `home` | Kennung der Wohnung (A, B…) | — | `logement` |
| `floor_area` | Wohnfläche der Wohnung | m² | `surface_m2` |
| `price` | Verkaufspreis | k€ | `prix_kEUR` |
| `sale` | Kennung des neuen Verkaufs (V1 bis V4) | — | `vente` |

## Die Rechnungen der Lektionen, in Python

Diese Zellen wiederholen die Rechnungen der Lektionen 3.1 bis 3.6.

In [ ]:
# Lektion 3.1 — zwei Geraden, zwei Fehlermaße (Gleichungen 3.1 und 3.2)
def mae(a, b):
    return np.mean(np.abs(y - (a * x + b)))

def mse(a, b):
    return np.mean((y - (a * x + b)) ** 2)

for name, a, b in [("Einleitung", 3, 30), ('Kollege', 3.3, 14)]:
    print(f'{name:10s} MAE = {mae(a, b):.2f} k€   MSE = {mse(a, b):.2f} k€²   RMSE = {np.sqrt(mse(a, b)):.2f} k€')

In [ ]:
# Lektion 3.2 — die Verlustkurve, b fest auf 30 k€ (Gleichung 3.3)
slopes = np.round(np.arange(2.5, 3.45, 0.1), 1)
print(pd.Series([round(mse(a, 30), 2) for a in slopes], index=slopes))
A = np.linspace(2.4, 3.6, 241)
plt.figure(figsize=(6, 3))
plt.plot(A, [mse(a, 30) for a in A], color=BLUE)
plt.scatter(slopes, [mse(a, 30) for a in slopes], color='black', zorder=3)
plt.xlabel('Steigung a (k€ pro m²)'); plt.ylabel('Verlust L(a) (k€²)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Lektion 3.3 — Sekanten, Ableitung (Gleichung 3.4) und zwei Aktualisierungen (Gleichung 3.5)
def derivative_a(a, b=30):
    e = y - (a * x + b)
    return -2 / len(x) * np.sum(x * e)

for h in (0.1, 0.01, 0.001):
    print(f'Sekante h = {h}: {(mse(2.9 + h, 30) - mse(2.9, 30)) / h:.1f}')
print('exakte Ableitung bei 2,9:', round(derivative_a(2.9), 1))

a, eta = 2.5, 0.00006
for stage in range(3):
    print(f'Schritt {stage}: a = {a}, L = {mse(a, 30):.2f}, L\'(a) = {derivative_a(a):.2f}')
    a = round(a - eta * derivative_a(a), 3)    # auf Tausendstel gerundet, wie im Buch

In [ ]:
# Lektion 3.4 — der Gradient (Gleichung 3.6) und ein Schritt für beide Koeffizienten (Gleichung 3.7)
def gradient(a, b):
    e = y - (a * x + b)
    return np.array([-2 / len(x) * np.sum(x * e), -2 / len(x) * np.sum(e)])

g = gradient(2.5, 30)
print('Gradient bei (2,5 ; 30):', g.round(2))
print('nach einem Schritt:', (np.array([2.5, 30]) - 0.00006 * g).round(4))

a, b = 2.5, 30.0
for k in range(1, 100001):
    a, b = np.array([a, b]) - 0.00006 * gradient(a, b)
    if k in (10, 1000, 100000):
        print(f'nach {k} Schritten: a = {a:.4f}, b = {b:.4f}, L = {mse(a, b):.2f}')
a_ls, b_ls = np.polyfit(x, y, 1)
print(f'Talboden: a = {a_ls:.3f}, b = {b_ls:.2f}, L = {mse(a_ls, b_ls):.2f}')
print(f'G, Rechenbeispiel: {a_ls * x_G + b_ls:.0f} k€ vorhergesagt für {y_G} k€ beobachtet')

In [ ]:
# Lektion 3.5 — zwei Abstiege mit denselben Startvorhersagen (Gleichung 3.8)
mu, sigma = x.mean(), x.std()          # std() teilt durch n, wie in Kapitel 1
z = (x - mu) / sigma
print(f'mu = {mu:.0f} m², sigma = {sigma:.3f} m², z = {z.round(2)}')

def descent_m2(eta, n_steps, a=2.5, b=30.0):
    for _ in range(n_steps):
        a, b = np.array([a, b]) - eta * gradient(a, b)
    return a, b

def descent_std(eta, n_steps, a=2.5, b=30.0):
    w, c = a * sigma, b + a * mu        # Gleichung 2.7: gleiche Vorhersagen am Start
    for _ in range(n_steps):
        e = y - (w * z + c)
        w, c = w + eta * 2 / len(x) * np.sum(z * e), c + eta * 2 / len(x) * np.sum(e)
    return w / sigma, c - w / sigma * mu  # Gleichung 3.8: zurück zu (a ; b)

print('Schritte | m², η = 0,000 06           | standardisiert, η = 0,3')
a, b = 2.5, 30.0
done = 0
for k in (0, 1, 5, 10, 1000, 100000):
    a, b = descent_m2(0.00006, k - done, a, b); done = k
    a_s, b_s = descent_std(0.3, min(k, 1000))
    print(f'{k:>8} | a = {a:.3f} b = {b:6.2f} L = {mse(a, b):8.2f} | '
          f'a = {a_s:.3f} b = {b_s:6.2f} L = {mse(a_s, b_s):8.2f}')

# Grenz-Lernraten: darüber divergiert der Abstieg (etwa 0,000 22 in m², 1 standardisiert)
for eta in (0.000216, 0.000217):
    print(f'm², η = {eta}: Verlust nach 400 Schritten = {mse(*descent_m2(eta, 400)):.4g} k€²')
for eta in (0.99, 1.01):
    print(f'standardisiert, η = {eta}: Verlust nach 400 Schritten = {mse(*descent_std(eta, 400)):.4g} k€²')

In [ ]:
# Lektion 3.6 — Ausgleichsgerade der kleinsten Quadrate (Gleichung 3.9) und R² (Gleichung 3.10)
cov = np.mean((x - x.mean()) * (y - y.mean()))
a_ls = cov / x.var()
b_ls = y.mean() - a_ls * x.mean()
print(f'cov = {cov:.1f} m²·k€, var = {x.var():.2f} (m²)², a = {a_ls:.3f} k€/m², b = {b_ls:.2f} k€')

def r2_lesson(observed, predicted, reference):
    """R² = 1 − MSE der Regel / MSE der konstanten Vorhersage « reference »."""
    return 1 - np.mean((observed - predicted) ** 2) / np.mean((observed - reference) ** 2)

y_mean = y.mean()                      # mittlerer Preis von A bis F: die Referenz für R² auf A bis F
print(f'A bis F: MSE konstant = {np.mean((y - y_mean) ** 2):.2f}, MSE Gerade = {mse(a_ls, b_ls):.2f}, '
      f'R² = {r2_lesson(y, a_ls * x + b_ls, y_mean):.3f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.3f}')

# Die drei Geraden, dasselbe Maß, auf A bis F, dann auf V1 bis V4 (Lektion 3.6)
lines = {'kleinste Quadrate': (a_ls, b_ls), 'Einleitung': (3, 30), 'Kollege': (3.3, 14)}
for name, (a, b) in lines.items():
    e_af, e_v = y - (a * x + b), yv - (a * xv + b)
    print(f'{name:16s} A bis F: MSE = {np.mean(e_af ** 2):6.2f}  MAE = {np.mean(np.abs(e_af)):5.2f}   |   '
          f'V1 bis V4: MSE = {np.mean(e_v ** 2):6.2f}  MAE = {np.mean(np.abs(e_v)):5.2f}   Abweichungen {e_v.round(2)}')
yv_bis = np.array([132, 215, 254, 312.0])   # falls V4 für 312 k€ verkauft worden wäre
for name in ('kleinste Quadrate', 'Kollege'):
    a, b = lines[name]
    print(f'V4 zu 312 k€: {name} MSE = {np.mean((yv_bis - (a * xv + b)) ** 2):.2f}')
print(f'G, Rechenbeispiel: {a_ls * x_G + b_ls:.2f} k€ vorhergesagt, Abweichung {y_G - (a_ls * x_G + b_ls):.2f} k€')

## Aufgabe 3.1 — Ein Diagramm ablesen: zwei Maße, zwei Minima

*Daten D0.1. Begriffe: MAE und MSE (Lektion 3.1), Verlustkurve (Lektion 3.2), Median (Lektion 1.3).* Die Steigung ist auf 3 k€/m² festgelegt. Beantworten Sie zuerst die Fragen a) bis e), indem Sie die Abbildung im Buch ablesen.

### a) bis e) Ihre Ablesungen

*Ihre Antworten:* …

### f) Im Notebook

In [ ]:
# ZU ERGÄNZEN: MSE und MAE der Geraden 3x + b auf A bis F, für einen Wert von b
def mse_b(b):
    return None   # ZU ERGÄNZEN

def mae_b(b):
    return None   # ZU ERGÄNZEN

In [ ]:
# Überprüfung, dann Darstellung der beiden Kurven
if mse_b(30) is None or mae_b(30) is None:
    print('Ergänzen Sie mse_b und mae_b.')
else:
    print('mse_b und mae_b:', 'richtig' if np.isclose(mse_b(30), 344 / 6) and np.isclose(mae_b(30), 44 / 6) else 'bitte prüfen')
    B = np.linspace(15, 45, 601)
    fig, (h, low) = plt.subplots(2, 1, figsize=(6, 4.5), sharex=True)
    h.plot(B, [mse_b(b) for b in B], color=BLUE); h.set_ylabel('MSE (k€²)'); h.grid(alpha=0.4)
    low.plot(B, [mae_b(b) for b in B], color=ORANGE); low.set_ylabel('MAE (k€)'); low.grid(alpha=0.4)
    low.set_xlabel('Achsenabschnitt b (k€)')
    plt.show()
    print('Knickstellen der MAE-Kurve (r_i sortiert):', np.sort(y - 3 * x))

In [ ]:
# ZU ERGÄNZEN: Median (Gleichung 1.4) und Mittelwert der r_i = y_i − 3 x_i
median_r = None
mean_r = None

In [ ]:
# Überprüfung
for name, yours, truth in [('Median', median_r, 29.5), ('Mittelwert', mean_r, 182 / 6)]:
    print(f'{name} der r:', 'noch nicht ausgefüllt' if yours is None else ('richtig' if np.isclose(yours, truth) else 'bitte prüfen'))
if median_r is not None and mae_b(30) is not None:
    print(f'MAE am Median: {mae_b(median_r):.2f} k€ (auf dem Plateau); '
          f'MSE am Mittelwert: {mse_b(mean_r):.2f} k€² (Minimum)')

*Ihre Schlussfolgerung:* …

## Aufgabe 3.2 — Zwei Schritte Gradientenabstieg

*Daten D0.1. Begriffe: mittlerer quadratischer Fehler (Lektion 3.1), Ableitung und Aktualisierung (Lektion 3.3).* Die Steigung ist auf 3 k€/m² festgelegt. Gesucht ist der Achsenabschnitt $b$. Bearbeiten Sie zuerst die Fragen a) bis d) auf Papier.

### a) Vorbereiten

In [ ]:
# ZU ERGÄNZEN: r_i = y_i − 3 x_i für die sechs Wohnungen (NumPy-Array)
r = None

In [ ]:
# Überprüfung
if r is None:
    print('Ergänzen Sie die vorherige Zelle.')
else:
    print('r:', 'richtig' if np.allclose(r, [22, 37, 23, 35, 24, 41]) else 'bitte prüfen')

### b) und c) Auf Papier

Tragen Sie hier Ihre Ergebnisse ein: Verluste in k€², Werte von $b$ in k€.

In [ ]:
# ZU ERGÄNZEN: Ihre von Hand berechneten Ergebnisse
L_b0 = None       # Verlust bei b0 = 20
derivative_b0 = None # L'(20)
b1 = None
L_b1 = None
b2 = None
L_b2 = None

In [ ]:
# Überprüfung (unabhängige Berechnung)
def _L(b): return np.mean((y - 3 * x - b) ** 2)
def _d(b): return -2 / 6 * np.sum(y - 3 * x - b)
_b1 = 20 - 0.3 * _d(20); _b2 = _b1 - 0.3 * _d(_b1)
expected_values = {'L_b0': (L_b0, _L(20)), 'derivative_b0': (derivative_b0, _d(20)), 'b1': (b1, _b1),
            'L_b1': (L_b1, _L(_b1)), 'b2': (b2, _b2), 'L_b2': (L_b2, _L(_b2))}
for name, (yours, truth) in expected_values.items():
    status = 'noch nicht ausgefüllt' if yours is None else ('richtig' if abs(yours - truth) < 0.01 else 'bitte prüfen')
    print(f'{name}: {status}')

### d) Interpretieren

*Ihre Antwort:* …

### e) Im Notebook

In [ ]:
# ZU ERGÄNZEN
def loss(b):
    """MSE der Geraden y = 3x + b auf A bis F (Gleichung 3.2)."""
    return None   # ZU ERGÄNZEN

def derivative(b):
    """L'(b) = −(2/n) Σ e_i (Gleichung 3.6)."""
    return None   # ZU ERGÄNZEN

<details><summary><b>Hinweis</b> (klicken Sie, wenn Sie nicht weiterkommen)</summary>

Berechnen Sie zuerst das Array der Abweichungen `e = y - (3 * x + b)`. `np.mean(e ** 2)` liefert den Verlust und `np.sum(e)` die Summe der Abweichungen.

</details>

In [ ]:
def descent(b0, eta, n_steps):
    """Gibt die Liste der b und die Liste der Verluste zurück, von Schritt 0 bis Schritt n_steps."""
    bs, losses = [b0], [loss(b0)]
    for _ in range(n_steps):
        bs.append(bs[-1] - eta * derivative(bs[-1]))   # Gleichung 3.5
        losses.append(loss(bs[-1]))
    return bs, losses

if loss(20) is None or derivative(20) is None:
    print('Ergänzen Sie loss und derivative.')
else:
    bs, losses = descent(20, 0.3, 2)
    print('zwei Schritte:', np.round(bs, 3), np.round(losses, 2), '→',
          'richtig' if np.allclose(bs, [20, 26.2, 28.68]) else 'bitte prüfen')
    bs, losses = descent(20, 0.3, 20)
    plt.figure(figsize=(6, 2.8))
    plt.plot(losses, marker='o', color=BLUE)
    plt.xlabel('Schritt'); plt.ylabel('Verlust (k€²)'); plt.grid(alpha=0.4)
    plt.show()
    print('b nach 20 Schritten:', round(bs[-1], 3), '  Mittelwert der r:', round(np.mean(y - 3 * x), 3))

### f) Vorhersagen, dann überprüfen: $\eta = 1{,}2$

*Ihre Vorhersage, vor dem Ausführen:* …

In [ ]:
if loss(20) is not None and derivative(20) is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))
    for ax, eta in zip(axes, (0.3, 1.2)):
        bs, losses = descent(20, eta, 6)
        ax.plot(bs, marker='o', color=ORANGE if eta > 1 else BLUE)
        ax.axhline(np.mean(y - 3 * x), color='black', linewidth=0.8, linestyle='--')
        ax.set_title(f'η = {eta}', fontsize=10); ax.set_xlabel('Schritt'); ax.set_ylabel('b (k€)')
        print(f'η = {eta} :', np.round(bs, 2))
    plt.tight_layout(); plt.show()

### g) Rechenbeispiel zu G

In [ ]:
# ZU ERGÄNZEN: vorhergesagter Preis für G mit Steigung 3 und b2, dann vorzeichenbehaftete Abweichung (k€)
price_G = None
gap_G = None

In [ ]:
# Überprüfung
if price_G is None or gap_G is None:
    print('Ergänzen Sie die vorherige Zelle.')
else:
    print('G:', 'richtig' if np.isclose(price_G, 214.68) and np.isclose(gap_G, -5.68) else 'bitte prüfen')

*Ihr Kommentar zu G:* …

## Aufgabe 3.3 — Was bedeutet „99 %“?

*Daten D0.1 und D3.1. Begriffe: Bestimmtheitsmaß, Anpassung und Bewertung (Lektion 3.6).* Bearbeiten Sie zuerst die Fragen a) bis f) auf Papier.

### a) bis f) Auf Papier

*Ihre Antworten:* …

### g) Im Notebook

In [ ]:
# ZU ERGÄNZEN: MSE von Vorhersagen, auf den übergebenen Verkäufen (Gleichung 3.2)
def sales_mse(observed, predicted):
    """observed und predicted: NumPy-Arrays gleicher Länge."""
    return None   # ZU ERGÄNZEN

In [ ]:
# Überprüfung: die Rangfolge aus Lektion 3.6, dann R² auf A bis F (Referenz: mittlerer Preis von A bis F)
if sales_mse(y, y) is None:
    print('Ergänzen Sie die Funktion sales_mse.')
else:
    a_ls = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
    b_ls = y.mean() - a_ls * x.mean()
    expected_values = {'kleinste Quadrate': ((a_ls, b_ls), 45.36, 93.29), 'Einleitung': ((3, 30), 57.33, 134.25),
                'Kollege': ((3.3, 14), 65.11, 70.73)}
    for name, ((a, b), att_af, att_v) in expected_values.items():
        e1, e2 = sales_mse(y, a * x + b), sales_mse(yv, a * xv + b)
        print(f'{name:16s} A bis F: {e1:6.2f}   V1 bis V4: {e2:6.2f} →',
              'richtig' if abs(e1 - att_af) < 0.006 and abs(e2 - att_v) < 0.006 else 'bitte prüfen')
    r2_af = 1 - sales_mse(y, a_ls * x + b_ls) / sales_mse(y, np.full(6, y.mean()))
    print(f'R² auf A bis F = {r2_af:.4f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.4f} →',
          'richtig' if np.isclose(r2_af, np.corrcoef(x, y)[0, 1] ** 2) else 'bitte prüfen')

## Aufgabe 3.4 — Ein Abstieg mit standardisierten Flächen

*Daten D0.1. Begriffe: Gradient (Lektion 3.4), Standardisierung und Rückumrechnung (Lektion 3.5), Ausgleichsgerade der kleinsten Quadrate (Lektion 3.6).*

### a) Auf Papier

*Ihre Rechnung:* …

### b) Mittelwert, Standardabweichung und standardisierte Werte

In [ ]:
# ZU ERGÄNZEN: Mittelwert und Standardabweichung der Flächen (Division durch n), dann die sechs z-Werte
mu = None
sigma = None
z = None

In [ ]:
# Überprüfung
if z is None:
    print('Ergänzen Sie die vorherige Zelle.')
else:
    print('z-Werte:', 'richtig' if np.isclose(np.mean(z), 0) and np.isclose(np.mean(z ** 2), 1) and np.isclose(mu, 64) else 'bitte prüfen')

### c) Der Gradient in der standardisierten Schreibweise

In [ ]:
# ZU ERGÄNZEN: die beiden partiellen Ableitungen (Lektion 3.5), als Array [dL/dw, dL/dc] zurückgegeben
def standardized_gradient(w, c):
    return None   # ZU ERGÄNZEN

In [ ]:
# Überprüfung am Startpunkt (gleiche Vorhersagen wie (2,5 ; 30))
if z is None or standardized_gradient(1, 1) is None:
    print('Ergänzen Sie die Zellen b) und c).')
else:
    w0, c0 = 2.5 * sigma, 30 + 2.5 * mu
    g0 = standardized_gradient(w0, c0)
    print(f'Start: w0 = {w0:.2f}, c0 = {c0:.0f}, gradient = {g0.round(2)} →',
          'richtig' if np.allclose(g0, [-29.683, -64.667], atol=0.001) else 'bitte prüfen')

### d) Dreißig Schritte, dann zurück zu den ursprünglichen Einheiten

In [ ]:
# ZU ERGÄNZEN: 30 Aktualisierungen mit eta = 0,3 ab (w0 ; c0), dann a und b (Gleichung 3.8)
a_std = None
b_std = None

In [ ]:
# Überprüfung: Gleichung 3.9 und np.polyfit
a_ls = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
b_ls = y.mean() - a_ls * x.mean()
print(f'Gleichung 3.9: a = {a_ls:.3f}, b = {b_ls:.2f}; np.polyfit: {np.round(np.polyfit(x, y, 1), 3)}')
if a_std is None:
    print('Ergänzen Sie die vorherige Zelle.')
else:
    print('erhaltene Gerade:', 'richtig' if np.allclose([a_std, b_std], [a_ls, b_ls], atol=1e-4) else 'bitte prüfen')

### e) und f) Der Verlust der beiden Abstiege, dann die Anzahl der Schritte

In [ ]:
# Die beiden Abstiege aus Lektion 3.5, Schritt für Schritt (nichts zu ergänzen)
def path_m2(n_steps, eta=0.00006):
    a, b, losses = 2.5, 30.0, []
    for _ in range(n_steps + 1):
        losses.append(np.mean((y - a * x - b) ** 2))
        e = y - a * x - b
        a, b = a + eta * 2 / 6 * np.sum(x * e), b + eta * 2 / 6 * np.sum(e)
    return losses

def path_std(n_steps, eta=0.3):
    zz = (x - x.mean()) / x.std()
    w, c, losses = 2.5 * x.std(), 30 + 2.5 * x.mean(), []
    for _ in range(n_steps + 1):
        losses.append(np.mean((y - w * zz - c) ** 2))
        e = y - w * zz - c
        w, c = w + eta * 2 / 6 * np.sum(zz * e), c + eta * 2 / 6 * np.sum(e)
    return losses

plt.figure(figsize=(6, 3))
plt.semilogy(path_m2(30), marker='o', color=ORANGE, label='Flächen in m², η = 0,000 06')
plt.semilogy(path_std(30), marker='s', color=BLUE, label='standardisierte Flächen, η = 0,3')
plt.axhline(45.364, color='black', linestyle='--', linewidth=0.8)
plt.xlabel('Anzahl der Schritte'); plt.ylabel('Verlust (k€²)'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# f) Anzahl der Schritte, bis der Verlust weniger als 0,1 k€² über dem Minimum liegt (einige Sekunden)
L_min = np.mean((y - (a_ls * x + b_ls)) ** 2)

def n_steps(X, a, b, eta, limit=10 ** 6):
    for k in range(1, limit + 1):
        e = y - a * X - b
        a, b = a + eta * 2 / 6 * np.sum(X * e), b + eta * 2 / 6 * np.sum(e)
        if np.mean((y - a * X - b) ** 2) - L_min < 0.1:
            return k

losses_std = path_std(30)
print('standardisierte Flächen, η = 0,3:', next(k for k, v in enumerate(losses_std) if v - L_min < 0.1), 'Schritte')
print('Flächen in m², η = 0,000 06:      ', n_steps(x, 2.5, 30.0, 0.00006), 'Schritte')

*Ihre Interpretation:* …

## Aufgabe 3.5 — Vorhersagen, dann überprüfen: Flächen in Einheiten von zehn m²

*Daten D0.1. Begriffe: Maßstabswechsel (Lektion 2.5), Gradient (Lektion 3.4), Standardisierung (Lektion 3.5).*

### a) bis d) Ihre Vorhersagen, vor dem Ausführen

*a):* …

*b):* …

*c):* …

*d):* …

### e) Überprüfen

In [ ]:
# ZU ERGÄNZEN: die Flächen von A bis F in Einheiten von zehn m²
x_tens = None

In [ ]:
# Überprüfung, dann die Messungen (sonst nichts zu ergänzen)
if x_tens is None:
    print('Ergänzen Sie die vorherige Zelle.')
else:
    print('x_tens:', 'richtig' if np.allclose(x_tens, [3.2, 4.5, 5.4, 7.0, 8.5, 9.8]) else 'bitte prüfen')
    print('a) Ausgleichsgerade (Steigung, Achsenabschnitt):', np.round(np.polyfit(x_tens, y, 1), 2))
    e0 = y - (25 * x_tens + 30)
    print(f'b) Quotient bei Schritt 0: {np.sum(x_tens * e0) / np.sum(e0):.2f} Einheiten von zehn m²')
    for eta in (0.021, 0.022):
        a, b = 25.0, 30.0
        for _ in range(200):
            e = y - a * x_tens - b
            a, b = a + eta * 2 / 6 * np.sum(x_tens * e), b + eta * 2 / 6 * np.sum(e)
        print(f'c) η = {eta}: Verlust nach 200 Schritten = {np.mean((y - a * x_tens - b) ** 2):.4g} k€²')
    L_min = np.mean((y - np.polyval(np.polyfit(x, y, 1), x)) ** 2)
    a, b = 25.0, 30.0
    for k in range(1, 10 ** 6):
        e = y - a * x_tens - b
        a, b = a + 0.006 * 2 / 6 * np.sum(x_tens * e), b + 0.006 * 2 / 6 * np.sum(e)
        if np.mean((y - a * x_tens - b) ** 2) - L_min < 0.1:
            break
    print(f'd) η = 0,006: {k} Schritte bis weniger als 0,1 k€² über dem Minimum')

### f) Interpretieren

*Ihre Antwort:* …

## Aufgabe 3.6 — Herausforderung: Kreise auf der Karte

*Daten D0.1. Begriffe: Standardisierung (Lektionen 1.4 und 3.5), partielle Ableitungen und Gradient (Lektion 3.4).* Die Fragen a) bis f) werden auf Papier bearbeitet. Das Notebook überprüft Ihre Ergebnisse.

### a) bis f) Auf Papier

*Ihre Beweise:* …

### Im Notebook überprüfen

In [ ]:
zz = (x - x.mean()) / x.std()
print(f'a) Summe der z = {np.sum(zz):.1e}, Summe der z² = {np.sum(zz ** 2):.4f}')

In [ ]:
# ZU ERGÄNZEN: m = (1/n) Σ z_i y_i, dann ein Abstiegsschritt mit Lernrate eta ab (w ; c)
m = None

def step(w, c, eta):
    """Gibt (w, c) nach einer Aktualisierung zurück (Gleichung 3.7 mit z statt x)."""
    return None   # ZU ERGÄNZEN

In [ ]:
# Überprüfung von b) und d)
if m is None or step(1, 1, 0.1) is None:
    print('Ergänzen Sie die vorherige Zelle.')
else:
    print(f'b) m = {m:.2f} →', 'richtig' if np.isclose(m, np.mean((x - x.mean()) * (y - y.mean())) / x.std()) else 'bitte prüfen')
    w0, c0 = 2.5 * x.std(), 30 + 2.5 * x.mean()
    w1, c1 = step(w0, c0, 0.5)
    print(f'd) ein Schritt mit η = 0,5: (w ; c) = ({w1:.2f} ; {c1:.2f}) →',
          'richtig' if np.allclose([w1, c1], [m, y.mean()]) else 'bitte prüfen')
    for eta in (1.0, 1.05):
        w, c = w0, c0
        sequence = []
        for _ in range(4):
            w, c = step(w, c, eta)
            sequence.append(round(c, 2))
        print(f'   η = {eta}: c nach 1, 2, 3, 4 Schritten = {sequence}')
    W, Cc = np.meshgrid(np.linspace(40, 100, 200), np.linspace(180, 260, 200))
    Z = np.mean([(yi - W * zi - Cc) ** 2 for zi, yi in zip(zz, y)], axis=0)
    plt.figure(figsize=(4.5, 4.5))
    plt.contour(W, Cc, Z, levels=[50, 100, 200, 400, 800, 1600], colors=BLUE)
    plt.gca().set_aspect('equal')
    plt.xlabel('w (k€ pro Standardabweichung)'); plt.ylabel('c (k€)'); plt.title('e) Kreise'); plt.show()